# install requirements:

In [0]:
%pip install -r /Volumes/workspace/default/real-estate/requirements.txt

In [0]:
%pip install gspread google-auth

# restart kernel to apply requirements:

In [0]:
%restart_python

# Run the imot.bg sitemap discovery 

## fiter by slug required_slug_keywords=["prodava-kashta", "varna"]

In [0]:
import sys

sys.path.append("/Volumes/workspace/default/real-estate/code")

import imot_bg_sitemap_discovery

houses_for_sale_in_varna = imot_bg_sitemap_discovery.discover_listing_id_url_pairs(
    required_slug_keywords=["prodava-kashta", "varna"]
)

print(f"total matched: {len(houses_for_sale_in_varna)}")

imot_bg_sitemap_discovery.save_pairs_to_file(houses_for_sale_in_varna)

# Run the scraper based on the sitemaps
## expected time 1–1.5 hours for a full run of 2510 listing URLs and that is just for Varna houses for sale

In [0]:
import sys

sys.path.append("/Volumes/workspace/default/real-estate/code")

import imot_bg_sitemap_discovery
import imot_bg_scraper

listing_id_url_pairs = imot_bg_sitemap_discovery.load_pairs_from_file()

imot_bg_scraper.scrape_listing_urls(listing_id_url_pairs)

# check some sample data

In [0]:
%skip
import json

detail_path = "/Volumes/workspace/default/real-estate/raw/detail_snapshot.jsonl"
history_path = "/Volumes/workspace/default/real-estate/raw/price_history.jsonl"

with open(detail_path, encoding="utf-8") as f:
    detail_rows = [json.loads(line) for line in f]

with open(history_path, encoding="utf-8") as f:
    history_rows = [json.loads(line) for line in f]

print("Current listings:", len(detail_rows))
print("Price history records:", len(history_rows))

print("\nExample listing:")
print(json.dumps(detail_rows[0], ensure_ascii=False, indent=2))

print("\nExample price history:")
print(json.dumps(history_rows[0], ensure_ascii=False, indent=2))

# Create Bronze tables for listings and price history

In [0]:
from pyspark.sql import functions as F

detail_path = "/Volumes/workspace/default/real-estate/raw/detail_snapshot.jsonl"

df = spark.read.json(detail_path)

df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.bronze_imot_listings")

price_history_path = "/Volumes/workspace/default/real-estate/raw/price_history.jsonl"

price_history_df = spark.read.json(price_history_path)

price_history_df = price_history_df.withColumn(
    "observed_at",
    F.to_timestamp("observed_at")
)

price_history_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.bronze_imot_price_history")


# test: query the delta bronze tables

In [0]:
%skip
%sql
SELECT *
FROM workspace.default.bronze_imot_listings
LIMIT 20;



In [0]:
%skip
%sql
SELECT *
FROM workspace.default.bronze_imot_price_history
ORDER BY observed_at DESC
LIMIT 20;

# Bronze → Silver transformation (listings)

Bronze contains the scraped data in a raw, mostly source-like format.
In Silver, the data is cleaned, standardized, and converted into analysis-ready columns.

Main changes:

- Convert price and property measurements from strings to numeric values
- Extract area, plot area, and floor from raw text
- Convert gas and district heating values into usable boolean fields
- Standardize construction type
- Split location information into region, location type, and location name
- Classify locations such as villages, towns, localities, resorts, and neighborhoods
- Extract listing edit date and view count from listing information
- Calculate price per square meter
- Remove temporary raw/helper columns used during transformation

The goal of Silver is to preserve the useful information from Bronze while making the data consistent and easy to use for analysis and Gold tables.

In [0]:
from pyspark.sql import functions as F


bronze_listings = spark.table(
    "workspace.default.bronze_imot_listings"
)

bronze_price_history = spark.table(
    "workspace.default.bronze_imot_price_history"
)


silver_listings = (
    bronze_listings

    .select(
        "listing_id",
        "url",
        "agency_name",
        "currency",

        # Convert invalid/non-numeric prices to NULL
        F.expr("try_cast(price AS double)").alias("price"),

        "location_text",
        "description",
        "feature_tags",
        "listing_info_text",

        F.col("property_parameters.Площ").alias("area_raw"),
        F.col("property_parameters.Двор").alias("plot_area_raw"),
        F.col("property_parameters.Етаж").alias("floor_raw"),
        F.col("property_parameters.Газ").alias("gas_raw"),
        F.col("property_parameters.ТEЦ").alias("district_heating_raw"),
        F.col("property_parameters.Строителство").alias("construction_raw")
    )

    # Convert property area to numeric square metres
    .withColumn(
        "area_sqm",
        F.regexp_extract(
            "area_raw",
            r"([0-9]+(?:[.,][0-9]+)?)",
            1
        ).cast("double")
    )

    .withColumn(
        "plot_area_sqm",
        F.regexp_extract(
            "plot_area_raw",
            r"([0-9]+(?:[.,][0-9]+)?)",
            1
        ).cast("double")
    )

    # Extract the first number from the floor
    
    # 3       -> 3
    # 3 от 3  -> 3
    # 2 от 3  -> 2
    .withColumn(
        "floor",
        F.regexp_extract(
            "floor_raw",
            r"^(\d+)",
            1
        ).cast("integer")
    )

    # Convert gas information to boolean
    
    # ДА -> true
    # НЕ -> false
    # Other values -> NULL
    .withColumn(
        "has_gas",
        F.when(
            F.upper(F.trim("gas_raw")) == "ДА",
            True
        )
        .when(
            F.upper(F.trim("gas_raw")) == "НЕ",
            False
        )
        .otherwise(None)
    )

    # Convert district heating information to boolean
    
    # ДА -> true
    # НЕ -> false
    # Other values -> NULL
    .withColumn(
        "has_district_heating",
        F.when(
            F.upper(F.trim("district_heating_raw")) == "ДА",
            True
        )
        .when(
            F.upper(F.trim("district_heating_raw")) == "НЕ",
            False
        )
        .otherwise(None)
    )

    # Preserve the actual heating type
    
    # Example:
    # ДА
    # НЕ
    # Лок.отопл.
    .withColumn(
        "district_heating_type",
        F.trim("district_heating_raw")
    )

    # Clean construction type
    
    # Тухла, -> Тухла
    .withColumn(
        "construction_type",
        F.trim(
            F.regexp_replace(
                "construction_raw",
                r",+$",
                ""
            )
        )
    )

    # Extract region from location
    
    # област Варна -> Варна
    # град Варна   -> Варна
    .withColumn(
        "region",
        F.regexp_extract(
            "location_text",
            r"^(?:област|град)\s+([^,]+)",
            1
        )
    )

    # Extract location type
    
    # с.   -> село
    # гр.  -> град
    # м-т  -> местност
    # к.к.  -> курортен комплекс
    # anything else -> квартал/район
    .withColumn(
        "location_type",
        F.when(
            F.instr("location_text", "м-т ") > 0,
            "местност"
        )
        .when(
            F.instr("location_text", "с. ") > 0,
            "село"
        )
        .when(
            F.instr("location_text", "гр. ") > 0,
            "град"
        )
        .when(
            F.instr("location_text", "к.к. ") > 0,
            "курортен комплекс"
        )
        .otherwise("квартал/район")
    )

    # Extract the actual location name
    
    # област Варна, с. Новаково
    # -> Новаково
    #
    # град Варна, м-т Долна Трака
    # -> Долна Трака
    .withColumn(
        "location_name",
        F.trim(
            F.regexp_replace(
                F.element_at(
                    F.split("location_text", ","),
                    -1
                ),
                r"^(с\.|гр\.|м-т)\s*",
                ""
            )
        )
    )

    # Extract the complete last-edited date/time only when
    # the expected Bulgarian format is present
    
    # Example:
    # Коригирана в 7:51 на 24 август, 2026 год.
    .withColumn(
        "last_edited_match",
        F.regexp_extract(
            "listing_info_text",
            r"Коригирана в [0-9]{1,2}:[0-9]{2} на [0-9]{1,2} [^,]+, [0-9]{4} год\.",
            0
        )
    )

    .withColumn(
        "last_edited_time",
        F.when(
            F.col("last_edited_match") != "",
            F.regexp_extract(
                "listing_info_text",
                r"Коригирана в ([0-9]{1,2}:[0-9]{2})",
                1
            )
        )
    )

    .withColumn(
        "last_edited_day",
        F.when(
            F.col("last_edited_match") != "",
            F.regexp_extract(
                "listing_info_text",
                r"на ([0-9]{1,2}) [^,]+, [0-9]{4} год",
                1
            )
        )
    )

    .withColumn(
        "last_edited_month",
        F.when(
            F.col("last_edited_match") != "",
            F.regexp_extract(
                "listing_info_text",
                r"на [0-9]{1,2} ([^,]+), [0-9]{4} год",
                1
            )
        )
    )

    .withColumn(
        "last_edited_year",
        F.when(
            F.col("last_edited_match") != "",
            F.regexp_extract(
                "listing_info_text",
                r", ([0-9]{4}) год",
                1
            )
        )
    )

    # Convert Bulgarian month names to month numbers
    .withColumn(
        "last_edited_month_number",
        F.when(F.col("last_edited_month") == "януари", "01")
        .when(F.col("last_edited_month") == "февруари", "02")
        .when(F.col("last_edited_month") == "март", "03")
        .when(F.col("last_edited_month") == "април", "04")
        .when(F.col("last_edited_month") == "май", "05")
        .when(F.col("last_edited_month") == "юни", "06")
        .when(F.col("last_edited_month") == "юли", "07")
        .when(F.col("last_edited_month") == "август", "08")
        .when(F.col("last_edited_month") == "септември", "09")
        .when(F.col("last_edited_month") == "октомври", "10")
        .when(F.col("last_edited_month") == "ноември", "11")
        .when(F.col("last_edited_month") == "декември", "12")
    )

    # Build timestamp only when all required components exist
    .withColumn(
        "last_edited_at",
        F.when(
            (F.col("last_edited_year").isNotNull()) &
            (F.col("last_edited_month_number").isNotNull()) &
            (F.col("last_edited_day").isNotNull()) &
            (F.col("last_edited_time").isNotNull()),
            F.to_timestamp(
                F.concat(
                    F.col("last_edited_year"),
                    F.lit("-"),
                    F.col("last_edited_month_number"),
                    F.lit("-"),
                    F.lpad(F.col("last_edited_day"), 2, "0"),
                    F.lit(" "),
                    F.col("last_edited_time")
                ),
                "yyyy-MM-dd H:mm"
            )
        )
    )

    # Extract number of listing views
    
    # Example:
    # Обявата е посетена 1471 пъти.
    .withColumn(
        "views",
        F.regexp_extract(
            "listing_info_text",
            r"посетена\s+([0-9]+)\s+пъти",
            1
        ).cast("long")
    )

    # Calculate price per square metre
    .withColumn(
        "price_per_sqm",
        F.when(
            (F.col("price").isNotNull()) &
            (F.col("area_sqm").isNotNull()) &
            (F.col("area_sqm") > 0),
            F.col("price") / F.col("area_sqm")
        )
    )

    # Remove temporary helper columns
    .drop(
        "area_raw",
        "plot_area_raw",
        "floor_raw",
        "gas_raw",
        "district_heating_raw",
        "construction_raw",
        "last_edited_match",
        "last_edited_time",
        "last_edited_day",
        "last_edited_month",
        "last_edited_year",
        "last_edited_month_number"
    )
)

# drop raw fields
silver_listings = silver_listings.drop(
    "url",
    "location_text",
    "listing_info_text",

)
# Inspect Silver

# display(
#     silver_listings.limit(50)
# )

# save silver 

silver_listings.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.silver_imot_listings")



In [0]:
%skip
spark.sql("""
SELECT *
FROM workspace.default.silver_imot_listings
LIMIT 20
""").display()

# Bronze → Silver transformation (price history)

- Removed the url column because it isn't needed for price-history analysis.
- Converted price from its raw value into a numeric double.
- Converted observed_at from text into a proper timestamp.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

bronze_price_history = spark.table(
    "workspace.default.bronze_imot_price_history"
)

silver_price_history = (
    bronze_price_history
    .select(
        "listing_id",
        F.expr("try_cast(price AS double)").alias("price"),
        "currency",
        F.to_timestamp("observed_at").alias("observed_at"),
        "url",
    )
)

listing_window = Window.partitionBy("listing_id")

silver_price_history = silver_price_history.withColumn(
    "observed_count",
    F.count("listing_id").over(listing_window)
)

# Overwrite the Silver table with the new schema
silver_price_history.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.silver_imot_price_history")




In [0]:
%skip
spark.sql("""
SELECT *
FROM workspace.default.silver_imot_price_history
LIMIT 10
""").display()

# Silver → Gold Transformation 

### The Silver layer contains cleaned property and price-history data, while the Gold layer contains aggregated market statistics, agency statistics, and price-change metrics.

## Source Silver Tables

The Gold transformations use two Silver tables:

- `silver_imot_listings`
  - Contains the current cleaned listing data.
  - Includes price, location, property characteristics, agency,
    views, area, and price per square metre.

- `silver_imot_price_history`
  - Contains historical price observations for each listing.
  - Includes listing ID, price, currency, and observation timestamp.

## Gold Tables Created

### 1. `gold_market_by_location`

This table provides aggregated market statistics by location.

The data is grouped by:

- Region
- Location name
- Location type
- Construction type

The table calculates:

- Number of listings
- Average price
- Median price
- Average price per square metre
- Median price per square metre
- Average property area

This table is intended to be the main source for market analysis
and Power BI dashboard visualisations.

### 2. `gold_agency_summary`

This table provides summary statistics for real-estate agencies.

The data is grouped by agency and calculates:

- Number of listings
- Average listing price
- Average listing views

This table can be used to analyse agency activity and create
agency-focused Power BI visualisations.

### 3. `gold_price_changes`

This table summarises price movement for each listing using the
historical price observations.

For each listing, it calculates:

- First observed price
- Latest observed price
- Absolute price change
- Percentage price change
- First observation date
- Latest observation date
- Number of price observations
- Number of days the listing has been tracked

The table is designed to become more useful over time as additional
price observations are collected by the scraper.

## Gold Layer Purpose

The Gold layer provides business-ready data instead of raw or
technical data structures.


In [0]:
from pyspark.sql import functions as F

silver_listings = spark.table("workspace.default.silver_imot_listings")
silver_price_history = spark.table("workspace.default.silver_imot_price_history")

# 1. Market summary by location 

gold_market_by_location = (
    silver_listings
    .filter(F.col("price").isNotNull() & F.col("price_per_sqm").isNotNull())
    .groupBy("region", "location_name", "location_type", "construction_type")
    .agg(
        F.count("*").alias("listing_count"),
        F.round(F.avg("price"), 0).alias("avg_price"),
        F.round(F.expr("percentile_approx(price, 0.5)"), 0).alias("median_price"),
        F.round(F.avg("price_per_sqm"), 2).alias("avg_price_per_sqm"),
        F.round(F.expr("percentile_approx(price_per_sqm, 0.5)"), 2).alias("median_price_per_sqm"),
        F.round(F.avg("area_sqm"), 1).alias("avg_area_sqm"),
    )
    .orderBy(F.desc("listing_count"))
)

gold_market_by_location.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_market_by_location")


# 2. Agency summary 

gold_agency_summary = (
    silver_listings
    .filter(F.col("agency_name").isNotNull() & F.col("price").isNotNull())
    .groupBy("agency_name")
    .agg(
        F.count("*").alias("listing_count"),
        F.round(F.avg("price"), 0).alias("avg_price"),
        F.round(F.avg("views"), 0).alias("avg_views"),
    )
    .orderBy(F.desc("listing_count"))
)

gold_agency_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_agency_summary")


from pyspark.sql import functions as F
from pyspark.sql import Window

silver_price_history = spark.table("workspace.default.silver_imot_price_history")

# Every dthe scraper ran (derived from the data itself)
scrape_dates = (
    silver_price_history
    .select(F.to_date("observed_at").alias("scrape_date"))
    .distinct()
)

listings = silver_price_history.select("listing_id").distinct()

# One row per listing per scrape date (the "spine")
spine = listings.crossJoin(scrape_dates)

# Real price-change events, collapsed to one row per listing per date
price_events = (
    silver_price_history
    .withColumn("scrape_date", F.to_date("observed_at"))
    .groupBy("listing_id", "scrape_date")
    .agg(F.max("price").alias("price"))
)

spine_with_price = spine.join(price_events, ["listing_id", "scrape_date"], "left")

fill_window = (
    Window.partitionBy("listing_id")
    .orderBy("scrape_date")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

gold_price_timeline = (
    spine_with_price
    .withColumn(
        "price",
        F.last("price", ignorenulls=True).over(fill_window)
    )
    .filter(F.col("price").isNotNull())
)

# Keep only listings where price actually varied at least once
listing_variation = (
    gold_price_timeline
    .groupBy("listing_id")
    .agg(
        F.min("price").alias("min_price"),
        F.max("price").alias("max_price"),
    )
    .filter(F.col("min_price") != F.col("max_price"))
    .select("listing_id")
)

gold_price_timeline_changed = gold_price_timeline.join(
    listing_variation, "listing_id", "inner"
)

# One representative URL per listing
listing_urls = (
    silver_price_history
    .groupBy("listing_id")
    .agg(F.first("url", ignorenulls=True).alias("url"))
)

gold_price_timeline_changed = gold_price_timeline_changed.join(
    listing_urls, "listing_id", "left"
)

gold_price_timeline_changed.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.gold_price_timeline_changed")

# Test: query gold_market_by_location

In [0]:
%skip
spark.sql("""
SELECT *
FROM workspace.default.gold_market_by_location
ORDER BY listing_count DESC
LIMIT 20
""").display()

# Test: query gold_agency_summary

In [0]:
%skip
spark.sql("""
SELECT *
FROM workspace.default.gold_agency_summary
ORDER BY listing_count DESC
LIMIT 20
""").display()

# test: query gold_price_changes

### this will be useful after several scrapings and wil catch anyy price changes

<pre>
scraper runs
   ↓
current price compared with previous price
   ↓
price changed?
   ├─ NO  → don't add a history record
   └─ YES → add new record to price_history.jsonl
                    ↓
             Silver price history
                    ↓
             Gold price changes
             </pre>


In [0]:
%skip
spark.sql("""
SELECT *
FROM workspace.default.gold_price_changes
ORDER BY price_change_pct ASC
LIMIT 100
""").display()

spark.sql("""
SELECT *
FROM workspace.default.gold_price_changes
WHERE observation_count > 1
ORDER BY price_change_pct ASC
LIMIT 100
""").display()

post to google sheets - price changes

In [0]:
import json
import pandas as pd
import gspread
from google.oauth2.service_account import Credentials

# ---------- Config ----------
KEY_PATH = "/Volumes/workspace/default/real-estate/secrets/our-axon-509919-m8-8ff8833d225c.json"
DATE_PATTERN = "m/d/yyyy"   # use "mm/dd/yyyy" for leading zeros

# Databricks table -> Google Sheet ID (the first tab of each sheet is used)
JOBS = {
    "workspace.default.gold_price_timeline_changed": "192yfEcDJBdG-g8uINodc0IremZ0Ihc_JH4htRII-qAQ",
    "workspace.default.gold_agency_summary":         "1dh-9E2RQKzBzwvBorfxNwDklnUwCkdMIwojjdTaUNlg",
    "workspace.default.gold_market_by_location":     "1XzIFasqFo1J6oVRlefch0-OQHlTjdcVZLul9B3NPjzw",
}

# Per-table columns that are stored as text but should be treated as dates
EXTRA_DATE_COLUMNS = {
    # "workspace.default.gold_agency_summary": ["scrape_date"],
}

# ---------- Auth ----------
with open(KEY_PATH) as f:
    sa_info = json.load(f)

scopes = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive",
]
creds = Credentials.from_service_account_info(sa_info, scopes=scopes)
gc = gspread.authorize(creds)


def push_table(table, sheet_id):
    sh = gc.open_by_key(sheet_id)
    worksheet = sh.get_worksheet(0)   # first tab, whatever it is named

    df = spark.table(table).toPandas()
    extra = EXTRA_DATE_COLUMNS.get(table, [])

    # Detect and convert date columns
    date_cols = []
    for col in df.columns:
        s = df[col]
        non_null = s.dropna()
        if pd.api.types.is_datetime64_any_dtype(s):
            is_date = True
        elif s.dtype == "object" and len(non_null) > 0:
            is_date = all(hasattr(x, "isoformat") for x in non_null)
        else:
            is_date = False

        if is_date or col in extra:
            df[col] = pd.to_datetime(s, errors="coerce").dt.strftime("%Y-%m-%d")
            date_cols.append(col)

    # Make everything JSON-serializable
    df = df.astype(object).where(df.notnull(), "")
    values = [df.columns.tolist()] + df.values.tolist()

    # Overwrite in place (same file, same ID), USER_ENTERED parses real dates
    worksheet.update(values=values, range_name="A1", value_input_option="USER_ENTERED")
    worksheet.resize(rows=len(values), cols=len(values[0]))

    # Format all date columns in one request
    if date_cols:
        formats = []
        for col in date_cols:
            letter = gspread.utils.rowcol_to_a1(1, df.columns.get_loc(col) + 1)[:-1]
            formats.append({
                "range": f"{letter}2:{letter}",
                "format": {"numberFormat": {"type": "DATE", "pattern": DATE_PATTERN}},
            })
        worksheet.batch_format(formats)

    print(f"{table}: wrote {len(values) - 1} rows, date columns: {date_cols}")


for table, sheet_id in JOBS.items():
    try:
        push_table(table, sheet_id)
    except Exception as e:
        print(f"FAILED {table}: {e}")